# Multi-Dataset PCB Component & Defect Detection Pipeline
### Supported Datasets: WACV 2019, Kaggle FICS-PCB, PKU-Market PCB Defect, & DeepPCB

This notebook provides a unified pipeline to download, clean, and process both PCB Component Detection and PCB Defect Detection datasets from Kaggle:
1. **Automatic Dataset Downloader**: Pulls WACV, FICS-PCB, PKU-Market, and DeepPCB datasets using `kagglehub`.
2. **Data Inspection & Cleaning**: Filters out invalid bounding boxes, corrupt images, and tiny noise.
3. **Pretrained SAM Point Extractor**: Passes box prompts into Meta's `sam_vit_b.pth` to generate exact `(x, y)` polygon points.
4. **CSV & YOLO Exporter**: Saves polygon points to both **`polygon_points.csv`** and YOLO-seg `.txt` label files.
5. **Polygon Data Augmentation**: Applies spatial flips, rotations, and HSV color jitter while transforming `(x, y)` points.

In [ ]:
# Step 1: Install Dependencies
!pip install torch torchvision opencv-python matplotlib pandas kagglehub git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os
import cv2
import torch
import random
import json
import pandas as pd
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor
import kagglehub

# Step 2: Download & Initialize Pretrained SAM Weights
sam_checkpoint = Path(r"C:\Users\ANAGHA\sam_vit_b.pth")
sam_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not sam_checkpoint.exists():
    print(f"Downloading SAM pretrained weights to {sam_checkpoint}...")
    sam_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(sam_url, str(sam_checkpoint))
    print("SAM weights download complete!")
else:
    print(f"SAM pretrained weights found at: {sam_checkpoint}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading SAM model on device: {device}...")
sam = sam_model_registry["vit_b"](checkpoint=str(sam_checkpoint))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
print("SAM Predictor ready!")

In [ ]:
# Step 3: Multi-Dataset Kaggle Downloader (FICS-PCB, PKU-Market, DeepPCB)
KAGGLE_DATASETS = {
    "fics_pcb": "ficslab/fics-pcb",                 # PCB Component Detection (9,912 images)
    "pku_pcb_defect": "akhatovar/pcb-defect-dataset", # PKU PCB Defect Dataset
    "deeppcb": "arnablaha/deeppcb"                   # DeepPCB Defect Dataset
}

def pull_kaggle_pcb_dataset(name="pku_pcb_defect"):
    slug = KAGGLE_DATASETS.get(name, name)
    print(f"Pulling Kaggle dataset '{name}' ({slug}) via kagglehub...")
    try:
        path = kagglehub.dataset_download(slug)
        print(f"SUCCESS: Kaggle dataset '{name}' downloaded to: {path}")
        return Path(path)
    except Exception as e:
        print(f"Note on Kaggle Download: {e}")
        return None

pku_defect_path = pull_kaggle_pcb_dataset("pku_pcb_defect")

In [ ]:
# Step 4: Extract Points and Save to CSV Function
def save_polygon_points_to_csv(extracted_records, output_csv_path):
    """
    Saves extracted SAM polygon (x, y) points into a clean Pandas CSV file.
    Columns: [image_name, instance_id, class_id, num_points, points_json, points_yolo_str]
    """
    rows = []
    for record in extracted_records:
        pts_pairs = [[record['points'][i], record['points'][i+1]] for i in range(0, len(record['points']), 2)]
        rows.append({
            'image_name': record['image_name'],
            'instance_id': record['instance_id'],
            'class_id': record['class_id'],
            'num_points': len(pts_pairs),
            'points_json': json.dumps(pts_pairs),
            'points_yolo_str': " ".join(map(str, record['points']))
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_csv_path, index=False)
    print(f"Saved {len(df)} SAM polygon instance records to CSV: {output_csv_path}")
    return df

In [ ]:
# Step 5: Demonstration of CSV Output Structure
sample_data = [
    {
        'image_name': 'pcb_001.jpg',
        'instance_id': 0,
        'class_id': 1,
        'points': [0.4411, 0.6679, 0.4333, 0.6843, 0.4328, 0.6906, 0.4410, 0.6920]
    },
    {
        'image_name': 'pcb_001.jpg',
        'instance_id': 1,
        'class_id': 2,
        'points': [0.1234, 0.5678, 0.1300, 0.5800, 0.1250, 0.5900, 0.1200, 0.5850]
    }
]

df_preview = save_polygon_points_to_csv(sample_data, "sample_polygon_points.csv")
df_preview.head()